In [1]:
import re
from pathlib import Path

import pandas as pd
from scipy.stats import ttest_ind

In [2]:
version = "eval_v2"
clfs = {"attn"}

paths = sorted(Path("output/subcortical").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|a424|schaefer400_tians3)_lr([-0-9e]+)_([0-9])", key)
    space, lr, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
state_table = pd.concat(tables, ignore_index=True)
print(state_table.shape)
state_table.head()

16
(52, 20)


,key,space,base_lr,run,model,repr,clf,dataset,ckpt,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,attn,hcpya_task21,best,12,0.000255,0.05,23,"[0.85, 1.0]",train,0.029348,0.993789,0.000586,0.992845,0.000736
1,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,attn,hcpya_task21,best,12,0.000255,0.05,23,"[0.85, 1.0]",validation,0.071022,0.977183,0.002281,0.973107,0.003096
2,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,attn,hcpya_task21,best,12,0.000255,0.05,23,"[0.85, 1.0]",test,0.101236,0.966865,0.002526,0.961609,0.003286
3,a424_lr3e-4_2,a424,0.0003,2,a424_mae,patch,attn,hcpya_task21,best,15,0.002940,0.05,38,"[9.8, 1.0]",train,0.000124,1.000000,0.000000,1.000000,0.000000
4,a424_lr3e-4_2,a424,0.0003,2,a424_mae,patch,attn,hcpya_task21,best,15,0.002940,0.05,38,"[9.8, 1.0]",validation,0.178503,0.976438,0.002338,0.970828,0.003356


In [3]:
state_summary = state_table.loc[state_table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["space", "base_lr", "run", "repr", "clf"], columns="dataset"
)
state_summary = state_summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
state_summary

acc                   acc_std  \
dataset                        hcpya_task21 nsd_cococlip hcpya_task21   
space              base_lr run                                          
a424               0.0003  1       0.966865          NaN     0.002526   
                           2       0.968849          NaN     0.002356   
                           3       0.969048          NaN     0.002447   
                           4       0.963294          NaN     0.002643   
schaefer400        0.0003  1       0.973810     0.272542     0.002263   
                           2       0.972817     0.277737     0.002331   
                           3       0.971429     0.275325     0.002372   
                           4       0.973214     0.274954     0.002301   
schaefer400_tians3 0.0003  1       0.971230          NaN     0.002515   
                           2       0.976984          NaN     0.002215   
                           3       0.978770          NaN     0.002096   
                           4       0.974405          NaN     0.002143   

                                             
dataset                        nsd_cococlip  
space              base_lr run               
a424               0.0003  1            NaN  
                           2            NaN  
                           3            NaN  
                           4            NaN  
schaefer400        0.0003  1       0.005209  
                           2       0.005153  
                           3       0.005251  
                           4       0.005252  
schaefer400_tians3 0.0003  1            NaN  
                           2            NaN  
                           3            NaN  
                           4            NaN

In [4]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [5]:
SPACE_NAMES = {
    "schaefer400": "Schaefer 400",
    "a424": "A424",
    "schaefer400_tians3": "Schaefer 400 + Tian S3",
}

SPACE_LRS = {
    "schaefer400": 3e-4,
    "a424": 3e-4,
    "schaefer400_tians3": 3e-4,
}

In [6]:
state_datasets = ["hcpya_task21", "nsd_cococlip"]

records = []

for (space, base_lr, run), row in state_summary.iterrows():
    if base_lr != SPACE_LRS[space]:
        continue
    record = {"space": space, "lr": base_lr, "run": run}
    for ds in state_datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

| space              |     lr |   run | HCP-YA Task21   | NSD COCO24   |
|:-------------------|-------:|------:|:----------------|:-------------|
| a424               | 0.0003 |     1 | 96.7 ± 0.3      | —            |
| a424               | 0.0003 |     2 | 96.9 ± 0.2      | —            |
| a424               | 0.0003 |     3 | 96.9 ± 0.2      | —            |
| a424               | 0.0003 |     4 | 96.3 ± 0.3      | —            |
| schaefer400        | 0.0003 |     1 | 97.4 ± 0.2      | 27.3 ± 0.5   |
| schaefer400        | 0.0003 |     2 | 97.3 ± 0.2      | 27.8 ± 0.5   |
| schaefer400        | 0.0003 |     3 | 97.1 ± 0.2      | 27.5 ± 0.5   |
| schaefer400        | 0.0003 |     4 | 97.3 ± 0.2      | 27.5 ± 0.5   |
| schaefer400_tians3 | 0.0003 |     1 | 97.1 ± 0.3      | —            |
| schaefer400_tians3 | 0.0003 |     2 | 97.7 ± 0.2      | —            |
| schaefer400_tians3 | 0.0003 |     3 | 97.9 ± 0.2      | —            |
| schaefer400_tians3 | 0.0003 |     4 | 97.4 ± 0.2 

In [7]:
records = []
for space, name in SPACE_NAMES.items():
    record = {"space": name}
    for ds in state_datasets:
        acc = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].mean()
        std = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

| space                  | HCP-YA Task21   | NSD COCO24   |
|:-----------------------|:----------------|:-------------|
| Schaefer 400           | 97.3 ± 0.1      | 27.5 ± 0.2   |
| A424                   | 96.7 ± 0.3      | —            |
| Schaefer 400 + Tian S3 | 97.5 ± 0.3      | —            |
\begin{tabular}{lll}
\toprule
space & HCP-YA Task21 & NSD COCO24 \\
\midrule
Schaefer 400 & 97.3 \mypm 0.1 & 27.5 \mypm 0.2 \\
A424 & 96.7 \mypm 0.3 & — \\
Schaefer 400 + Tian S3 & 97.5 \mypm 0.3 & — \\
\bottomrule
\end{tabular}



In [8]:
spaces = ["schaefer400", "schaefer400_tians3", "a424"]

test_results = {}

for ds in state_datasets:
    for ii in range(2):
        for jj in range(ii + 1, 3):
            space_a = spaces[ii]
            space_b = spaces[jj]
            acc_a = state_summary.loc[(space_a, SPACE_LRS[space_a]), ("acc", ds)].values
            acc_b = state_summary.loc[(space_b, SPACE_LRS[space_b]), ("acc", ds)].values
            result = ttest_ind(acc_a, acc_b, equal_var=False)
            result_fmt = f"t={result.statistic:.1f} p={result.pvalue:.1e}"
            test_results[space_a, space_b, ds] = {
                "statistic": result.statistic,
                "pvalue": result.pvalue,
            }
            print(ds, space_a, space_b, result_fmt)

hcpya_task21 schaefer400 schaefer400_tians3 t=-1.5 p=2.2e-01
hcpya_task21 schaefer400 a424 t=4.1 p=1.7e-02
hcpya_task21 schaefer400_tians3 a424 t=3.9 p=8.2e-03
nsd_cococlip schaefer400 schaefer400_tians3 t=nan p=nan
nsd_cococlip schaefer400 a424 t=nan p=nan
nsd_cococlip schaefer400_tians3 a424 t=nan p=nan


In [9]:
version = "eval_v2"
clfs = {"logistic"}

paths = sorted(Path("output/subcortical").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|a424|schaefer400_tians3)_lr([-0-9e]+)_([0-9])", key)
    space, lr, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
trait_table = pd.concat(tables, ignore_index=True)
print(trait_table.shape)
trait_table.head()

72
(14544, 17)


,key,space,base_lr,run,model,repr,clf,dataset,trial,C,split,acc,acc_std,f1,f1_std,bacc,bacc_std
0,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,logistic,aabc_age,NaN,0.005995,train,0.594488,0.021716,0.588052,0.022057,0.594755,0.021646
1,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,logistic,aabc_age,NaN,0.005995,test,0.365385,0.062906,0.351055,0.062947,0.356456,0.062602
2,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,logistic,aabc_age,1.0,0.005995,train,0.590551,0.021065,0.586064,0.021532,0.592426,0.020990
3,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,logistic,aabc_age,1.0,0.005995,test,0.461538,0.051828,0.411287,0.051609,0.459020,0.051101
4,a424_lr3e-4_1,a424,0.0003,1,a424_mae,patch,logistic,aabc_age,2.0,0.005995,train,0.586614,0.019925,0.581770,0.020067,0.587156,0.019897


In [10]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


trait_repr = "patch"

trait_summary = trait_table.query("split == 'test' and trial > 0").pivot_table(
    values=["acc", "bacc"],
    index=["space", "base_lr", "run", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
trait_summary = trait_summary.loc[
    (slice(None), slice(None), slice(None), trait_repr, "logistic")
].dropna(axis=1, how="all")
trait_summary

mean                                 \
                                     acc                                  
dataset                         aabc_age  aabc_sex  abide_dx adhd200_dx   
space              base_lr run                                            
a424               0.0003  1    0.442500  0.721273  0.634839   0.592308   
                           2    0.449231  0.726727  0.633468   0.582462   
                           3    0.436154  0.729273  0.643306   0.593846   
                           4    0.448654  0.724000  0.633145   0.600615   
schaefer400        0.0003  1    0.442115  0.723818  0.620323   0.589231   
                           2    0.442308  0.710364  0.624032   0.582308   
                           3    0.451154  0.723818  0.636290   0.587231   
                           4    0.434808  0.720909  0.637258   0.594923   
schaefer400_tians3 0.0003  1    0.439038  0.751818  0.617903   0.606154   
                           2    0.429423  0.740000  0.622177   0.590462   
                           3    0.436154  0.714000  0.614194   0.584308   
                           4    0.442692  0.735091  0.610968   0.593385   

                                                                          \
                                                          bacc             
dataset                        adni_ad_vs_cn ppmi_dx  aabc_age  aabc_sex   
space              base_lr run                                             
a424               0.0003  1        0.733659  0.6443  0.440888  0.711005   
                           2        0.731707  0.6453  0.447656  0.715020   
                           3        0.731220  0.6432  0.434583  0.717514   
                           4        0.739512  0.6549  0.447138  0.713349   
schaefer400        0.0003  1        0.730244  0.6506  0.440250  0.712643   
                           2        0.731707  0.6359  0.440366  0.699918   
                           3        0.747317  0.6379  0.449766  0.711298   
                           4        0.746829  0.6526  0.432308  0.708920   
schaefer400_tians3 0.0003  1        0.731463  0.6356  0.436822  0.742758   
                           2        0.741220  0.6390  0.426445  0.729973   
                           3        0.730000  0.6593  0.434808  0.701637   
                           4        0.738780  0.6499  0.440703  0.724042   

                                                     ...       sem             \
                                                     ...       acc              
dataset                         abide_dx adhd200_dx  ...  abide_dx adhd200_dx   
space              base_lr run                       ...                        
a424               0.0003  1    0.629732   0.575130  ...  0.003682   0.005469   
                           2    0.628025   0.566438  ...  0.004824   0.005202   
                           3    0.637831   0.576525  ...  0.003689   0.006028   
                           4    0.625935   0.581559  ...  0.003692   0.005904   
schaefer400        0.0003  1    0.614165   0.569040  ...  0.004031   0.005970   
                           2    0.617815   0.564870  ...  0.004167   0.006089   
                           3    0.631213   0.564112  ...  0.004311   0.005880   
                           4    0.631812   0.574821  ...  0.004219   0.005454   
schaefer400_tians3 0.0003  1    0.611155   0.585077  ...  0.004041   0.005553   
                           2    0.616928   0.569686  ...  0.004043   0.005099   
                           3    0.608892   0.562153  ...  0.003815   0.005956   
                           4    0.604989   0.574556  ...  0.004251   0.005331   

                                                                            \
                                                            bacc             
dataset                        adni_ad_vs_cn   ppmi_dx  aabc_age  aabc_sex   
space              base_lr run                                               
a424     

In [11]:
trait_datasets = ["abide_dx", "adhd200_dx", "adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex"]
trait_metric = "bacc"

records = []

for (space, base_lr, run), row in trait_summary.iterrows():
    if base_lr != SPACE_LRS[space]:
        continue
    record = {"space": space, "lr": base_lr, "run": run}
    for ds in trait_datasets:
        acc = row[("mean", trait_metric, ds)]
        std = row[("sem", trait_metric, ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

| space              |     lr |   run | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   |
|:-------------------|-------:|------:|:-----------|:-------------|:-----------|:-----------|:------------|:------------|
| a424               | 0.0003 |     1 | 63.0 ± 0.4 | 57.5 ± 0.5   | 62.0 ± 0.8 | 60.2 ± 0.4 | 44.1 ± 0.6  | 71.1 ± 0.6  |
| a424               | 0.0003 |     2 | 62.8 ± 0.5 | 56.6 ± 0.5   | 60.3 ± 0.8 | 60.0 ± 0.5 | 44.8 ± 0.6  | 71.5 ± 0.6  |
| a424               | 0.0003 |     3 | 63.8 ± 0.4 | 57.7 ± 0.6   | 61.3 ± 0.8 | 60.1 ± 0.4 | 43.5 ± 0.6  | 71.8 ± 0.6  |
| a424               | 0.0003 |     4 | 62.6 ± 0.4 | 58.2 ± 0.6   | 62.6 ± 0.8 | 61.3 ± 0.4 | 44.7 ± 0.7  | 71.3 ± 0.6  |
| schaefer400        | 0.0003 |     1 | 61.4 ± 0.4 | 56.9 ± 0.6   | 61.1 ± 0.7 | 61.2 ± 0.4 | 44.0 ± 0.7  | 71.3 ± 0.6  |
| schaefer400        | 0.0003 |     2 | 61.8 ± 0.4 | 56.5 ± 0.6   | 60.1 ± 0.8 | 60.2 ± 0.4 | 44.0 ± 0.7  | 70.0 ± 0.7  |
| schaefer400        | 0

In [12]:
for ds in trait_datasets:
    for ii in range(2):
        for jj in range(ii + 1, 3):
            space_a = spaces[ii]
            space_b = spaces[jj]
            acc_a = trait_summary.loc[
                (space_a, SPACE_LRS[space_a]), ("mean", trait_metric, ds)
            ].values
            acc_b = trait_summary.loc[
                (space_b, SPACE_LRS[space_b]), ("mean", trait_metric, ds)
            ].values
            result = ttest_ind(acc_a, acc_b, equal_var=False)
            result_fmt = f"t={result.statistic:.1f} p={result.pvalue:.1e}"
            test_results[space_a, space_b, ds] = {
                "statistic": result.statistic,
                "pvalue": result.pvalue,
            }
            print(ds, space_a, space_b, result_fmt)

abide_dx schaefer400 schaefer400_tians3 t=2.6 p=5.4e-02
abide_dx schaefer400 a424 t=-1.3 p=2.6e-01
abide_dx schaefer400_tians3 a424 t=-5.5 p=1.5e-03
adhd200_dx schaefer400 schaefer400_tians3 t=-0.9 p=4.3e-01
adhd200_dx schaefer400 a424 t=-1.7 p=1.5e-01
adhd200_dx schaefer400_tians3 a424 t=-0.4 p=7.4e-01
adni_ad_vs_cn schaefer400 schaefer400_tians3 t=1.0 p=3.9e-01
adni_ad_vs_cn schaefer400 a424 t=0.5 p=6.1e-01
adni_ad_vs_cn schaefer400_tians3 a424 t=-0.6 p=5.7e-01
ppmi_dx schaefer400 schaefer400_tians3 t=-0.1 p=9.2e-01
ppmi_dx schaefer400 a424 t=0.5 p=6.3e-01
ppmi_dx schaefer400_tians3 a424 t=0.8 p=4.6e-01
aabc_age schaefer400 schaefer400_tians3 t=1.3 p=2.5e-01
aabc_age schaefer400 a424 t=-0.4 p=7.0e-01
aabc_age schaefer400_tians3 a424 t=-1.8 p=1.2e-01
aabc_sex schaefer400 schaefer400_tians3 t=-1.8 p=1.5e-01
aabc_sex schaefer400 a424 t=-1.9 p=1.3e-01
aabc_sex schaefer400_tians3 a424 t=1.2 p=3.1e-01


In [14]:
records = []

for space in spaces:
    record = {"space": SPACE_NAMES[space]}
    for ds in trait_datasets:
        acc = trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].mean()
        std = trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    for ds in state_datasets:
        acc = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].mean()
        std = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
# print(result_fmt.to_latex(index=False).replace("±", "\mypm"))
print(re.sub("± ([.0-9]+)", r"\\mypm{\1}", result_fmt.to_latex(index=False)))

| space                  | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   | HCP-YA Task21   | NSD COCO24   |
|:-----------------------|:-----------|:-------------|:-----------|:-----------|:------------|:------------|:----------------|:-------------|
| Schaefer 400           | 62.4 ± 0.9 | 56.8 ± 0.5   | 62.1 ± 1.7 | 60.7 ± 0.9 | 44.1 ± 0.7  | 70.8 ± 0.6  | 97.3 ± 0.1      | 27.5 ± 0.2   |
| Schaefer 400 + Tian S3 | 61.0 ± 0.5 | 57.3 ± 1.0   | 61.2 ± 0.6 | 60.7 ± 0.6 | 43.5 ± 0.6  | 72.5 ± 1.7  | 97.5 ± 0.3      | —            |
| A424                   | 63.0 ± 0.5 | 57.5 ± 0.6   | 61.5 ± 1.0 | 60.4 ± 0.6 | 44.3 ± 0.6  | 71.4 ± 0.3  | 96.7 ± 0.3      | —            |
\begin{tabular}{lllllllll}
\toprule
space & ABIDE Dx & ADHD200 Dx & ADNI Dx & PPMI Dx & HCP-A Age & HCP-A Sex & HCP-YA Task21 & NSD COCO24 \\
\midrule
Schaefer 400 & 62.4 \mypm{0.9} & 56.8 \mypm{0.5} & 62.1 \mypm{1.7} & 60.7 \mypm{0.9} & 44.1 \mypm{0.7} & 70.8 \mypm{0.6} & 97.3 \mypm{0.1} 

In [15]:
test_table = pd.DataFrame.from_dict(test_results, orient="index")
test_table

statistic    pvalue
schaefer400        schaefer400_tians3 hcpya_task21   -1.474909  0.222574
                   a424               hcpya_task21    4.066850  0.016524
schaefer400_tians3 a424               hcpya_task21    3.943039  0.008235
schaefer400        schaefer400_tians3 nsd_cococlip         NaN       NaN
                   a424               nsd_cococlip         NaN       NaN
schaefer400_tians3 a424               nsd_cococlip         NaN       NaN
schaefer400        schaefer400_tians3 abide_dx        2.557917  0.054259
                   a424               abide_dx       -1.266049  0.263731
schaefer400_tians3 a424               abide_dx       -5.517582  0.001499
schaefer400        schaefer400_tians3 adhd200_dx     -0.863346  0.431841
                   a424               adhd200_dx     -1.680056  0.146850
schaefer400_tians3 a424               adhd200_dx     -0.356238  0.735735
schaefer400        schaefer400_tians3 adni_ad_vs_cn   0.973860  0.390358
                   a424               adni_ad_vs_cn   0.540875  0.612991
schaefer400_tians3 a424               adni_ad_vs_cn  -0.615665  0.566110
schaefer400        schaefer400_tians3 ppmi_dx        -0.102605  0.922161
                   a424               ppmi_dx         0.510471  0.630301
schaefer400_tians3 a424               ppmi_dx         0.787336  0.461121
schaefer400        schaefer400_tians3 aabc_age        1.280267  0.249002
                   a424               aabc_age       -0.402037  0.701885
schaefer400_tians3 a424               aabc_age       -1.830003  0.117009
schaefer400        schaefer400_tians3 aabc_sex       -1.811385  0.150973
                   a424               aabc_sex       -1.898066  0.125423
schaefer400_tians3 a424               aabc_sex        1.192834  0.314916

In [16]:
cutoff = 1e-4
cutoff = float(f"{cutoff:.0e}")
print(cutoff)
test_table.query(f"pvalue < {cutoff}")

0.0001


,,,statistic,pvalue


In [17]:
cutoff = 0.05 / len(test_table)
cutoff = float(f"{cutoff:.0e}")
print(cutoff)
test_table.query(f"pvalue < {cutoff}")

0.002


,,,statistic,pvalue
schaefer400_tians3,a424,abide_dx,-5.517582,0.001499
